Notebook: crm_ingest_interactions

Layer: Ingestion → Bronze
Source: CRM (JSON files from Unity Catalog Volume)

Description:

Ingests CRM interactions data using Auto Loader (incremental ingestion)
Handles semi-structured JSON (nested schema supported)
Adds ingestion metadata for traceability
Writes raw data to Bronze Delta table

Architecture: Volumes → Auto Loader → Bronze Delta


Step 1 — Imports

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

Step 2 — Define paths

In [0]:
source_path = "/Volumes/workspace/default/raw_data/crm/interactions/"
schema_location = "/Volumes/workspace/default/raw_data/schema/crm/interactions/"
checkpoint_location = "/Volumes/workspace/default/raw_data/checkpoints/crm/interactions/"

Step 3 — Read JSON + metadataColumns

In [0]:
import pyspark.sql.functions as F

df_crm_interactions_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(source_path)

        # Metadata
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_path"))
)

Step 4— Write to Bronze Delta table

In [0]:
(
    df_crm_interactions_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_location)
        .trigger(availableNow=True)
        .toTable("bronze.crm_interactions")
)

In [0]:
%sql
DESCRIBE bronze.crm_interactions 

In [0]:
%sql
SELECT * FROM bronze.crm_interactions LIMIT 10;

In [0]:
spark.table("bronze.crm_interactions").count()

In [0]:
display(spark.table("bronze.crm_customers"))